# Notebook 01 — FP32 Baseline Accuracy Measurement

## What does this notebook do?

Loads 4 models with ImageNet-pretrained weights and measures their Top-1 accuracy on the validation set.

**NO TRAINING.** Models are obtained ready-to-use from the torchvision library.

| Model | Expected FP32 Accuracy |
|---|---|
| ResNet-18 | ~69.8% |
| ResNet-50 | ~76.1% |
| MobileNetV2 | ~71.9% |

forward pass only, no training

## Prerequisite: HuggingFace Access

1. Create an account at [huggingface.co](https://huggingface.co)
2. On the [imagenet-1k](https://huggingface.co/datasets/ILSVRC/imagenet-1k) page, click "Agree and access repository" to request access to the dataset
3. Log in with your HuggingFace token in the cell below

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
shutil.copytree('/content/drive/MyDrive/imagenet_v2',
                '/content/mlcalib_v2', dirs_exist_ok=True)

import sys
os.chdir('/content/mlcalib_v2')
sys.path.insert(0, '/content/mlcalib_v2')
print('Klasörler:', os.listdir('.'))

Mounted at /content/drive
Klasörler: ['figures', 'results', 'notebooks', 'requirements.txt', 'src', 'config.py']


In [ ]:
!pip install datasets huggingface_hub scikit-learn matplotlib seaborn tqdm scipy -q

In [ ]:
# HuggingFace login for imagenet-1k
from huggingface_hub import login
login()

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

from config import MODELS, DEVICE
print(f'Device: {DEVICE}')
print(f'Modeller: {MODELS}')

GPU: NVIDIA A100-SXM4-40GB
Device: cuda
Modeller: ['resnet18', 'resnet50', 'efficientnet_b0', 'mobilenet_v2']


In [ ]:
from src.data_utils import get_val_loader

val_loader = get_val_loader(batch_size=64, max_samples=5000)
print(f'Val batch sayısı: {len(val_loader)}')

  Validation seti yükleniyor (5000 örnek)...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Val batch sayısı: 79


In [ ]:
# Accuracy for each model without quantization (FP32)
from src.model_utils import load_model, evaluate_model, save_fp32_baselines

baselines = {}
for model_name in MODELS:
    print(f'\n--- {model_name} ---')
    model = load_model(model_name, device=DEVICE)
    acc = evaluate_model(model, val_loader, DEVICE)
    baselines[model_name] = acc
    print(f'  FP32 Top-1 Accuracy: {acc:.2f}%')

df = save_fp32_baselines(baselines)
print('\nFP32 Baseline Tablosu:')
display(df)


--- resnet18 ---
  resnet18 yükleniyor (ImageNet pretrained)...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


  ✓ resnet18 yüklendi
  FP32 Top-1 Accuracy: 69.06%

--- resnet50 ---
  resnet50 yükleniyor (ImageNet pretrained)...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 242MB/s]


  ✓ resnet50 yüklendi
  FP32 Top-1 Accuracy: 75.88%

--- efficientnet_b0 ---
  efficientnet_b0 yükleniyor (ImageNet pretrained)...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


...

  FP32 Top-1 Accuracy: 77.36%

--- mobilenet_v2 ---
  mobilenet_v2 yükleniyor (ImageNet pretrained)...
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 226MB/s]

  ✓ mobilenet_v2 yüklendi


  FP32 Top-1 Accuracy: 71.12%
  ✓ FP32 baseline'lar kaydedildi: results/aggregated/fp32_baselines.csv

FP32 Baseline Tablosu:


,model,fp32_accuracy
0,resnet18,69.06
1,resnet50,75.88
2,efficientnet_b0,77.36
3,mobilenet_v2,71.12


In [ ]:
shutil.copytree('/content/mlcalib_v2/results',
                '/content/drive/MyDrive/imagenet_v2/results',
                dirs_exist_ok=True)
print('✓ Drive\'a kaydedildi.')

✓ Drive'a kaydedildi.
